# 第 22 课｜订单簿基础

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`22_youtube-1rJNDTqkucY/index.html`  
> **所属模块**：P1-M5 · 市场微观结构与执行

## 本节量化问题

一张订单簿如何决定立即成交价格和可见流动性？

## 准确定义与策略报告位置

最优买卖价形成报价价差，档位数量形成可见深度；订单簿不平衡只能描述当前排队状态。

- **核心关系**：中间价=(best bid+best ask)/2；价差=best ask-best bid；不平衡=(bid depth-ask depth)/(bid depth+ask depth)。
- **写入报告**：报价、深度、订单类型和流动性状态
- **P2 衔接**：P2 第 34–39、52 课：表格、时间对齐和异步数据。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 从 L2 快照计算价差、深度和不平衡

先按买卖方向分别找最优价；深度使用前 5 档可见数量，不能代表隐藏流动性。

In [2]:
book = pd.read_csv(DATA_DIR / "order_book.csv")
display(book)

bids = book.loc[book["side"] == "bid"].sort_values("price", ascending=False)
asks = book.loc[book["side"] == "ask"].sort_values("price", ascending=True)
best_bid = bids.iloc[0]["price"]
best_ask = asks.iloc[0]["price"]
mid = (best_bid + best_ask) / 2.0
spread_bps = (best_ask - best_bid) / mid * 10_000
imbalance = (bids["quantity"].sum() - asks["quantity"].sum()) / book["quantity"].sum()

display(pd.Series({"best_bid": best_bid, "best_ask": best_ask, "mid": mid, "spread_bps": spread_bps, "depth_imbalance": imbalance}))

,snapshot_time,side,level,price,quantity
0,2025-01-02 10:00:00+08:00,bid,1,99.99,300
1,2025-01-02 10:00:00+08:00,ask,1,100.01,270
2,2025-01-02 10:00:00+08:00,bid,2,99.98,420
3,2025-01-02 10:00:00+08:00,ask,2,100.02,400
4,2025-01-02 10:00:00+08:00,bid,3,99.97,540
5,2025-01-02 10:00:00+08:00,ask,3,100.03,530
6,2025-01-02 10:00:00+08:00,bid,4,99.96,660
7,2025-01-02 10:00:00+08:00,ask,4,100.04,660
8,2025-01-02 10:00:00+08:00,bid,5,99.95,780
9,2025-01-02 10:00:00+08:00,ask,5,100.05,790


best_bid            99.990000
best_ask           100.010000
mid                100.000000
spread_bps           2.000000
depth_imbalance      0.009346
dtype: float64

## 方法边界与阶段验收

可见挂单可能撤销，隐藏订单不可见；订单簿信号必须匹配真实时间顺序。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
